# Asymmetry, Example 2: a quantum Z4 symmetry

**Manuscript map.** Sec. III E, Fig. 6, Eqs. (59)-(64).

A single particle hops coherently and incoherently on a four-site ring.
The Lindblad generator commutes with cyclic translations and therefore
splits into four $\mathbb Z_4$ charge blocks. We construct two physical
density matrices with controlled block support and show that the state
concentrated in a faster sector restores symmetry first.

This implementation deliberately fixes eigenoperator phases by
hermitizing the selected modes. That avoids phase-dependent,
non-Hermitian states that can arise from the eigendecomposition
conventions of some SciPy versions.


In [ ]:
import sys
from pathlib import Path
import warnings

import matplotlib.pyplot as plt
import numpy as np
import scipy.linalg as la

_HERE = Path.cwd().resolve()
for _candidate in [_HERE, *_HERE.parents]:
    if (_candidate / "tools" / "notebook_utils.py").exists():
        _TOOLS_DIR = _candidate / "tools"
        break
else:
    raise RuntimeError(
        "Could not locate tools/notebook_utils.py from cwd or its parents."
    )
if str(_TOOLS_DIR) not in sys.path:
    sys.path.insert(0, str(_TOOLS_DIR))

import notebook_utils as nu

warnings.filterwarnings("ignore", category=RuntimeWarning)
np.set_printoptions(precision=5, suppress=True)
plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update({
    "figure.dpi": 120,
    "figure.figsize": (7.2, 4.2),
    "axes.spines.top": False,
    "axes.spines.right": False,
    "legend.frameon": False,
    "font.size": 11,
})

BLUE = "#394F87"
ORANGE = "#E8792E"
GREEN = "#3A7D44"
PURPLE = "#7C4D9E"


## 1. Covariant Lindbladian and irreducible tensor basis

The Hamiltonian is nearest-neighbor hopping with $J=1$. Biased left/right
jumps have rates $1\pm\varepsilon$, with $\varepsilon=1/4$.
In the tensor basis $\{T_\alpha^{(\mu)}\}$ of Eq. (59), covariance requires
the superoperator to be block diagonal in $\mu$.


In [ ]:
N = 4
EPSILON = 0.25
HOPPING = 1.0

L = nu.quantum_ring_liouvillian(N, EPSILON, HOPPING)
tensor_basis = nu.z4_fourier_tensor_basis(N)
B = np.column_stack([nu.vec(operator) for operator in tensor_basis])
L_blocks_basis = la.inv(B) @ L @ B

gram = B.conj().T @ B
block_mask = np.zeros_like(L_blocks_basis, dtype=bool)
for mu in range(N):
    block_mask[mu * N : (mu + 1) * N,
               mu * N : (mu + 1) * N] = True
leakage = la.norm(np.where(block_mask, 0, L_blocks_basis))

print(f"Basis orthonormality error: {la.norm(gram - np.eye(N**2)):.2e}")
print(f"Off-block Liouvillian norm: {leakage:.2e}")
assert leakage < 1e-10


In [ ]:
block_eigenvalues = {}
block_eigenvectors = {}
for mu in range(N):
    block = L_blocks_basis[
        mu * N : (mu + 1) * N,
        mu * N : (mu + 1) * N,
    ]
    values, vectors = la.eig(block)
    block_eigenvalues[mu] = values
    block_eigenvectors[mu] = vectors
    print(
        f"mu={mu}:",
        np.array([
            complex(round(value.real, 4), round(value.imag, 4))
            for value in values
        ]),
    )


## 2. Phase-stable slow- and fast-sector states

We mirror the design of Eq. (63):

- $\rho_1$ has modest support in the slow conjugate sectors
  $\mu=1,3$, whose eigenvalues have real part $-2$.
- $\rho_2$ has larger initial asymmetry but support in a fast,
  real-eigenvalue mode of the self-conjugate $\mu=2$ sector.

Each traceless Hermitian mode is added to the maximally mixed steady
state. Its amplitude is chosen as a fixed fraction of the positivity
boundary, so physicality is guaranteed independently of eigenvector phase.


In [ ]:
def embedded_eigenoperator(mu, column):
    coefficients = np.zeros(N**2, dtype=complex)
    coefficients[mu * N : (mu + 1) * N] = (
        block_eigenvectors[mu][:, column]
    )
    return (B @ coefficients).reshape(N, N, order="F")

# A slow mu=1 mode and its adjoint (which lies in mu=3).
slow_column = int(np.argmax(block_eigenvalues[1].imag))
slow_raw = embedded_eigenoperator(1, slow_column)
slow_mode = nu.hermitize(slow_raw + slow_raw.conj().T)
slow_mode -= np.trace(slow_mode) * np.eye(N) / N
slow_mode /= la.norm(slow_mode)

# The fastest real eigenmode in the self-conjugate mu=2 block.
real_columns = np.flatnonzero(
    np.abs(block_eigenvalues[2].imag) < 1e-8
)
fast_column = real_columns[
    np.argmin(block_eigenvalues[2][real_columns].real)
]
fast_mode = nu.hermitize(
    embedded_eigenoperator(2, int(fast_column))
)
fast_mode -= np.trace(fast_mode) * np.eye(N) / N
fast_mode /= la.norm(fast_mode)

def state_from_mode(mode, positivity_fraction):
    lower_bound = (1 / N) / (-la.eigvalsh(mode).min())
    return np.eye(N) / N + positivity_fraction * lower_bound * mode

rho_1 = state_from_mode(slow_mode, positivity_fraction=0.30)
rho_2 = state_from_mode(fast_mode, positivity_fraction=0.70)
for state in (rho_1, rho_2):
    nu.validate_density(state)

M0 = [
    nu.asymmetry_relative_entropy(state, nu.z4_twirl(state))
    for state in (rho_1, rho_2)
]
print("Initial asymmetry [rho_1, rho_2]:", M0)
print("Selected fast eigenvalue:",
      block_eigenvalues[2][fast_column])
assert M0[1] > M0[0]


## 3. Dynamics, sector norms, and crossing

For every snapshot we compute the relative entropy of asymmetry

$$
M[\rho(t)]=S(\rho(t)\Vert\mathcal G[\rho(t)])
=S(\mathcal G[\rho(t)])-S(\rho(t)),
$$

together with the trace norm $\|\rho^{(\mu)}(t)\|_1$ of each nonzero
mode.


In [ ]:
times = np.linspace(0, 2, 500)

def evolve_and_measure(rho0):
    asymmetry = []
    mode_norms = np.zeros((len(times), N))
    for index, time in enumerate(times):
        rho_t = (la.expm(L * time) @ nu.vec(rho0)).reshape(
            N, N, order="F"
        )
        rho_t = nu.normalize_density(rho_t)
        asymmetry.append(
            nu.asymmetry_relative_entropy(rho_t, nu.z4_twirl(rho_t))
        )
        for mu, mode in nu.z4_modes(rho_t).items():
            mode_norms[index, mu] = np.sum(la.svdvals(mode))
    return np.asarray(asymmetry), mode_norms

M_1, norms_1 = evolve_and_measure(rho_1)
M_2, norms_2 = evolve_and_measure(rho_2)
tau = nu.crossing_time(times, M_2, M_1)

print(f"Crossing time: {tau:.4f}")
assert np.isfinite(tau)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10.5, 4))

axes[0].semilogy(times, M_1, color=BLUE, lw=2,
                label=r"$M[\rho_1(t)]$: slow sectors")
axes[0].semilogy(times, M_2, color=ORANGE, lw=2,
                label=r"$M[\rho_2(t)]$: fast sector")
axes[0].axvline(tau, color="0.25", ls=":", lw=1)
axes[0].set(xlabel=r"$t$", ylabel=r"$M[\rho(t)]$",
            title=r"Quantum $Z_4$ Mpemba crossing")
axes[0].legend(fontsize=9)

for mu in range(1, N):
    if norms_1[:, mu].max() > 1e-10:
        axes[1].semilogy(
            times, norms_1[:, mu], color=BLUE,
            ls=["-", "--", ":"][mu - 1],
            label=fr"$\rho_1^{{({mu})}}$",
        )
    if norms_2[:, mu].max() > 1e-10:
        axes[1].semilogy(
            times, norms_2[:, mu], color=ORANGE,
            ls=["-", "--", ":"][mu - 1],
            label=fr"$\rho_2^{{({mu})}}$",
        )
axes[1].set(xlabel=r"$t$", ylabel=r"$\|\rho^{(\mu)}(t)\|_1$",
            title="Occupied symmetry sectors")
axes[1].legend(ncol=2, fontsize=9)
fig.tight_layout()
plt.show()


## Takeaway

Covariance makes the Liouvillian block diagonal in symmetry charge. The
initial amount of asymmetry does not determine the restoration speed by
itself: the relevant information is *where* that asymmetry sits. The
more asymmetric $\rho_2$ is concentrated in a block with a more negative
decay exponent and therefore crosses below $\rho_1$.
